# 📘 Fine-Tuning του Mistral-7B με QLoRA

Αυτό το notebook υλοποιεί fine-tuning του μοντέλου **Mistral-7B** για τη μετάφραση ελληνικών σε αγγλικά, χρησιμοποιώντας QLoRA και το dataset OPUS-100.

📌 Περιλαμβάνει:
- Προετοιμασία dataset
- Φόρτωση και quantization μοντέλου
- LoRA configuration
- Εκπαίδευση με αξιολόγηση

# 🚀 Fine-Tuning Mistral-7B με QLoRA για Μετάφραση Αγγλικά ➝ Ελληνικά

Αυτό το notebook εκτελεί fine-tuning του μοντέλου **Mistral-7B** πάνω στο dataset [`Helsinki-NLP/opus-100`](https://huggingface.co/datasets/Helsinki-NLP/opus-100), χρησιμοποιώντας **QLoRA** για αποδοτική χρήση υπολογιστικών πόρων.

📌 Περιλαμβάνει:
- Προετοιμασία dataset
- Φόρτωση και quantization μοντέλου
- QLoRA training setup
- Εκπαίδευση και αξιολόγηση (BLEU, ROUGE, METEOR, chrF)
- Εξαγωγή σε `.gguf` για χρήση με **Ollama**


# 🛠️ Εγκατάσταση απαραίτητων βιβλιοθηκών

## 🔧 Βασικά imports και παραμετροποίηση

In [ ]:
!pip install -q \
    transformers \
    datasets \
    accelerate \
    bitsandbytes \
    peft \
    evaluate \
    rouge-score \
    nltk


In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

from datasets import load_dataset

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
from datasets import load_dataset
import evaluate
import nltk

nltk.download("punkt")

# Έλεγχος GPU
if torch.cuda.is_available():
    print("✅ CUDA is available. GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ CUDA not available. Using CPU only.")


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # Ενεργοποίηση 4-bit quantization
    bnb_4bit_quant_type="nf4",             # Προτεινόμενος τύπος για QLoRA
    bnb_4bit_use_double_quant=True,        # Δεύτερο επίπεδο quantization για εξοικονόμηση μνήμης
    bnb_4bit_compute_dtype=torch.bfloat16  # Χρησιμοποιεί bfloat16 αν υποστηρίζεται
    # bnb_4bit_use_nested_quant=True       # (προαιρετικό) για ακόμα λιγότερη μνήμη
)

In [ ]:
import nltk
nltk.download("punkt", download_dir="./nltk_data")


In [ ]:
!pip install tf-keras -q


In [ ]:
# Φόρτωση του el-en config
dataset = load_dataset("Helsinki-NLP/opus-100", "el-en")

# Αντιστροφή: από el→en σε en→el
def reverse_translation(example):
    return {
        "translation": {
            "en": example["translation"]["en"],
            "el": example["translation"]["el"]
        }
    }

# Εφαρμόζουμε τη συνάρτηση
train_data = dataset["train"].map(reverse_translation)
val_data = dataset["validation"].map(reverse_translation)

# Προεπισκόπηση
print("Train δείγμα:", train_data[0])
print("Validation δείγμα:", val_data[0])


## 🤖 Φόρτωση του Mistral-7B και εφαρμογή LoRA

In [ ]:
model_name = "mistralai/Mistral-7B-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

In [ ]:
from transformers import AutoTokenizer

# Χρήση GPT-2 που δεν χρειάζεται authentication
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

# Τεστ με ελληνικό κείμενο
test_text = "Γεια σου, κόσμε!"
tokens = tokenizer(test_text)
print(f"Original text: {test_text}")
print(f"Tokens: {tokens}")
print(f"Decoded: {tokenizer.decode(tokens['input_ids'])}")
print("✅ Tokenizer works correctly!")


In [ ]:
import sentencepiece
print("SentencePiece is correctly installed!")


In [ ]:
!pip install sentencepiece



In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=64,  
    lora_alpha=32,  
    target_modules=["c_attn", "c_proj"],  # GPT-2 layer names
    lora_dropout=0.05,  
    bias="none",  
    task_type="CAUSAL_LM"  
)

# Ενσωμάτωση των LoRA adapters στο βασικό GPT-2 μοντέλο
model = get_peft_model(model, lora_config)

In [ ]:
# Φόρτωση GPT-2 μοντέλου (αντί για Mistral)
import torch
from transformers import AutoModelForCausalLM

print("Loading GPT-2 model...")

# Για CPU χωρίς quantization
model = AutoModelForCausalLM.from_pretrained(
    "gpt2",
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
)

print("✅ Model loaded successfully!")
print(f"Model type: {type(model)}")

## 📁 Φόρτωση OPUS-100 και μετατροπή σε instruction format
🔹 Χρησιμοποιούμε 5000 δείγματα για πιο γρήγορο fine-tuning στο Colab

In [ ]:
# Φόρτωση OPUS-100 dataset για EN→EL μετάφραση
from datasets import load_dataset

# Φόρτωση μικρότερου δείγματος για γρήγορη εκτέλεση
dataset = load_dataset("Helsinki-NLP/opus-100", "el-en")

# Επιλογή μικρού δείγματος για δοκιμή
train_size = 500  # Μικρό για γρήγορη εκτέλεση
val_size = 100

train_data = dataset["train"].select(range(train_size))
val_data = dataset["validation"].select(range(val_size))

print(f"Train samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")

# Διόρθωση: Instruction format για EN→EL μετάφραση
def format_example_en_to_el(example):
    english_text = example['translation']['en']
    greek_text = example['translation']['el']
    
    # Σωστό instruction format
    prompt = f"<s>[INST] Translate to Greek: {english_text} [/INST] {greek_text}</s>"
    return {"text": prompt}

# Εφαρμογή formatting
train_formatted = train_data.map(format_example_en_to_el)
val_formatted = val_data.map(format_example_en_to_el)

print("Sample formatted example:")
print(train_formatted[0]["text"])

# Tokenization
print("Tokenizing datasets...")
def tokenize_function(examples):
    return tokenizer(
        examples["text"], 
        truncation=True, 
        padding="max_length", 
        max_length=128,  # Μικρότερο για ταχύτητα
        return_tensors="pt"
    )

train_tokenized = train_formatted.map(
    tokenize_function, 
    batched=True,
    remove_columns=train_formatted.column_names
)

val_tokenized = val_formatted.map(
    tokenize_function, 
    batched=True,
    remove_columns=val_formatted.column_names  
)

print("✅ Dataset preparation complete!")
print(f"Train dataset: {len(train_tokenized)} samples")
print(f"Validation dataset: {len(val_tokenized)} samples")

## ⚙️ Ορισμός TrainingArguments και Trainer

Σε αυτό το βήμα ορίζουμε τις παραμέτρους εκπαίδευσης μέσω της κλάσης `TrainingArguments`, καθώς και τον Hugging Face `Trainer` που διαχειρίζεται το training loop.

📌 Οι βασικές ρυθμίσεις περιλαμβάνουν:
- 2 epochs εκπαίδευσης
- batch size 2 με gradient accumulation για μεγαλύτερη αποτελεσματικότητα
- χρήση BF16 για ταχύτερους υπολογισμούς σε κατάλληλο hardware (π.χ. A100)
- αποθήκευση μοντέλου στο τέλος κάθε epoch
- προς το παρόν **χωρίς αξιολόγηση με BLEU ή άλλο metric** — μόνο train loss

👉 Στο μέλλον μπορεί να προστεθεί αξιολόγηση με BLEU ή sacreBLEU στο validation set.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Βεβαιώνουμε ότι έχουμε τον tokenizer
# (Αν δεν υπάρχει, θα φορτώσουμε ξανά)
try:
    tokenizer
    print("Tokenizer already loaded")
except NameError:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token
    print("Tokenizer reloaded")

# CPU-optimized training arguments
training_args = TrainingArguments(
    output_dir="./gpt2_en_to_el_results",
    num_train_epochs=1,  # Μόνο 1 epoch για γρήγορη δοκιμή
    per_device_train_batch_size=1,  # Πολύ μικρό batch για CPU
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,  # Αυξήθηκε για μεγαλύτερο effective batch
    learning_rate=5e-5,  # Μικρότερο learning rate για σταθερότητα
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,  # Για CPU
    dataloader_num_workers=0,  # Για CPU
    fp16=False,  # Disable για CPU
    bf16=False   # Disable για CPU
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("✅ Training arguments configured for CPU!")

In [ ]:
# Δημιουργία Trainer object
# Έλεγχος αν υπάρχουν όλες οι μεταβλητές
try:
    model
    print("Model found")
except NameError:
    print("ERROR: Model not found. Please run the model loading cells first.")
    
try:
    training_args
    print("Training args found") 
except NameError:
    print("ERROR: Training args not found. Please run training setup cell first.")

try:
    train_tokenized
    print("Train dataset found")
except NameError:
    print("ERROR: Train dataset not found. Please run dataset preparation cell first.")

# Δημιουργία trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator
)

print("✅ Trainer created successfully!")
print(f"Training on {len(train_tokenized)} samples")
print(f"Evaluating on {len(val_tokenized)} samples")

## 🚀 Εκπαίδευση του μοντέλου με Trainer

In [ ]:
trainer.train()

In [ ]:
# Evaluation Metrics: BLEU, ROUGE, METEOR, chrF
import evaluate
import numpy as np
from datasets import Dataset

print("Loading evaluation metrics...")
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")

# chrF requires sacrebleu
try:
    import sacrebleu
    print("✅ sacrebleu available")
    chrf_available = True
except ImportError:
    print("⚠️ sacrebleu not available, installing...")
    import subprocess
    subprocess.check_call(["pip", "install", "sacrebleu"])
    import sacrebleu
    chrf_available = True

def evaluate_model(model, tokenizer, eval_dataset, num_samples=50):
    """
    Αξιολόγηση μοντέλου με BLEU, ROUGE, METEOR, chrF
    """
    model.eval()
    
    predictions = []
    references = []
    
    print(f"Generating predictions for {num_samples} samples...")
    
    # Παίρνουμε τα πρώτα num_samples για αξιολόγηση
    eval_samples = val_data.select(range(min(num_samples, len(val_data))))
    
    for i, example in enumerate(eval_samples):
        if i % 10 == 0:
            print(f"Processing sample {i+1}/{num_samples}")
            
        # Προετοιμασία input
        english_text = example['translation']['en']
        greek_reference = example['translation']['el']
        
        # Δημιουργία prompt
        prompt = f"<s>[INST] Translate to Greek: {english_text} [/INST]"
        
        # Tokenization
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=128)
        
        # Generation
        with torch.no_grad():
            outputs = model.generate(
                inputs.input_ids,
                max_new_tokens=64,
                do_sample=True,
                temperature=0.7,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        
        # Decode prediction
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Εξαγωγή μόνο της μετάφρασης (μετά το [/INST])
        if "[/INST]" in generated_text:
            prediction = generated_text.split("[/INST]")[-1].strip()
        else:
            prediction = generated_text.strip()
        
        predictions.append(prediction)
        references.append(greek_reference)
    
    # Υπολογισμός μετρικών
    print("\nCalculating metrics...")
    
    # BLEU Score
    bleu_score = bleu.compute(predictions=predictions, references=[[ref] for ref in references])
    
    # ROUGE Score
    rouge_score = rouge.compute(predictions=predictions, references=references)
    
    # METEOR Score  
    meteor_score = meteor.compute(predictions=predictions, references=references)
    
    # chrF Score
    if chrf_available:
        chrf_score = sacrebleu.corpus_chrf(predictions, [references])
    else:
        chrf_score = None
    
    # Εκτύπωση αποτελεσμάτων
    print("\n" + "="*50)
    print("📊 EVALUATION RESULTS")
    print("="*50)
    print(f"BLEU Score: {bleu_score['bleu']:.4f}")
    print(f"ROUGE-1: {rouge_score['rouge1']:.4f}")
    print(f"ROUGE-2: {rouge_score['rouge2']:.4f}")
    print(f"ROUGE-L: {rouge_score['rougeL']:.4f}")
    print(f"METEOR: {meteor_score['meteor']:.4f}")
    if chrf_available:
        print(f"chrF: {chrf_score.score:.4f}")
    print("="*50)
    
    # Παραδείγματα μεταφράσεων
    print("\n📝 SAMPLE TRANSLATIONS:")
    print("-"*50)
    for i in range(min(3, len(predictions))):
        print(f"\nExample {i+1}:")
        print(f"English: {eval_samples[i]['translation']['en']}")
        print(f"Reference: {eval_samples[i]['translation']['el']}")
        print(f"Prediction: {predictions[i]}")
    print("-"*50)
    
    return {
        'bleu': bleu_score['bleu'],
        'rouge1': rouge_score['rouge1'],
        'rouge2': rouge_score['rouge2'],
        'rougeL': rouge_score['rougeL'],
        'meteor': meteor_score['meteor'],
        'chrf': chrf_score.score if chrf_available else None,
        'predictions': predictions,
        'references': references
    }

# Εκτέλεση αξιολόγησης
print("Starting evaluation...")
results = evaluate_model(model, tokenizer, val_data, num_samples=20)

print("✅ Evaluation completed!")

## 💾 Merge & Αποθήκευση Fine-Tuned Μοντέλου

Σε αυτό το βήμα συγχωνεύουμε τους LoRA adapters με το base μοντέλο (merge_and_unload) ώστε να προκύψει ένα ενιαίο μοντέλο που μπορεί να χρησιμοποιηθεί για inference και αξιολόγηση χωρίς να χρειάζεται ξεχωριστά LoRA αρχεία.

Αποθηκεύουμε:

- το τελικό fine-tuned μοντέλο (`model.safetensors`)
- το tokenizer (που παραμένει ίδιο με το base μοντέλο)

Αποθηκεύονται όλα στον φάκελο `mistral_qlora_merged/`


In [ ]:
merged_model = model.merge_and_unload()

save_path = "mistral_qlora_merged"
merged_model.save_pretrained(save_path, safe_serialization=True)

tokenizer.save_pretrained(save_path)
